Mechanism Hypotheses:
1. Lateral exchange w/ surrounding water. For a cyclone moving south, water with low CHL enters and dilutes/reduces CHL concentration. 
2. Eddy-wind interaction. Ekman pumping drives upwelling in anticyclones and downwelling in cyclones. But if this is the case, then we should expect a change/response at the centers first.
3. Seasonal changes and mixed layer depth changes. 

In [ ]:
from pathlib import Path
from typing import cast
import sys
import warnings

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from matplotlib.axes import Axes
from matplotlib.cm import ScalarMappable
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
from matplotlib.figure import Figure
from matplotlib.ticker import MaxNLocator
from scipy.stats import binomtest, wilcoxon
from statsmodels.tools.sm_exceptions import ConvergenceWarning, SingularMatrixWarning

PROJECT_ROOT = Path('/Users/jerry/school/research/eddy-tracking')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from eddy_tracking.preprocess.tracks import PET_EPOCH
from eddy_tracking.packages.py_eddy_tracker.observations.tracking import TrackEddiesObservations

EXPERIMENT = 'gulf_stream_20240305_20260531'
N_BOOTSTRAP = 2000
N_AGE_BINS = 5
RANDOM_SEED = 2026
NEAR_AXIS_KM = 150
DATA_DIR = PROJECT_ROOT / 'data' / EXPERIMENT
polarity_names = ('cyclone', 'anticyclone')
target_classes = {'cyclone': 'NS', 'anticyclone': 'SN'}
polarity_colors = {'cyclone': '#2166ac', 'anticyclone': '#b2182b'}
polarity_labels = {'cyclone': 'Target cyclones', 'anticyclone': 'Target anticyclones'}
identity_columns = ['polarity', 'track_id']

movement = pd.read_parquet(DATA_DIR / 'silver/gulf_stream/eddy_movement.parquet')
plankton = pd.read_parquet(DATA_DIR / 'gold/eddy_plankton_table.parquet')
target_class = cast(pd.Series, movement['polarity']).map(target_classes)
movement['is_target'] = (
    movement['movement'].eq(target_class)
    | (
        movement['birth_distance_km'].abs().le(NEAR_AXIS_KM)
        & movement['death_side'].eq(target_class.str[1])
    )
)
analysis = cast(pd.DataFrame, plankton.merge(
    movement[identity_columns + ['is_target']], on=identity_columns, how='left',
).loc[lambda frame: frame['is_target']]).copy()
analysis['date'] = pd.to_datetime(analysis['date'])
analysis['eddy_key'] = analysis['polarity'] + ':' + analysis['track_id'].astype(str)
analysis['is_anticyclone'] = analysis['polarity'].eq('anticyclone').astype(int)
annual_angle = 2 * np.pi * (analysis['date'].dt.dayofyear - 1) / 365.25
analysis['season_sin'] = np.sin(annual_angle)
analysis['season_cos'] = np.cos(annual_angle)

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'mathtext.fontset': 'custom', 'mathtext.rm': 'Arial', 'mathtext.it': 'Arial:italic', 'mathtext.bf': 'Arial:bold',
    'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 8, 'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6, 'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False,
    'figure.dpi': 150, 'savefig.dpi': 300,
})

Are the changes that we're seeing the result of a few eddies with large increases/decreases or a consistent behavior across many eddies?

In [ ]:
bin_centers = (np.arange(N_AGE_BINS) + 0.5) / N_AGE_BINS
analysis['age_bin'] = np.minimum((analysis['age_frac'] * N_AGE_BINS).astype(int), N_AGE_BINS - 1)
eddy_bins = analysis.groupby(identity_columns + ['age_bin'])['CHL'].mean()
rng = np.random.default_rng(RANDOM_SEED)
drop_rows = []
curve_limits = [np.inf, -np.inf]
effect_limits = [np.inf, -np.inf]
loo_fig, loo_axes = cast(tuple[Figure, np.ndarray], plt.subplots(
    2, 2, figsize=(6.69, 4.8), sharex=True, sharey='row', layout='constrained',  # pyright: ignore[reportArgumentType]
    gridspec_kw={'height_ratios': (1.7, 1)},
))
for panel, polarity in enumerate(polarity_names):
    curve_ax = cast(Axes, loo_axes[0, panel])
    effect_ax = cast(Axes, loo_axes[1, panel])
    color = polarity_colors[polarity]
    table = eddy_bins.loc[polarity].unstack('age_bin').reindex(columns=range(N_AGE_BINS))
    matrix = table.to_numpy(dtype=float)  # (n_eddies, n_bins)
    finite = np.isfinite(matrix)
    counts = finite.sum(axis=0)
    sums = np.nansum(matrix, axis=0)
    full_mean = sums / counts
    without_one = (sums - np.where(finite, matrix, 0.0)) / (counts - finite)  # (n_eddies, n_bins)
    exclusion_effect = np.where(finite, without_one - full_mean, np.nan)
    curve_limits = [min(curve_limits[0], without_one.min()), max(curve_limits[1], without_one.max())]
    effect_limits = [min(effect_limits[0], np.nanmin(exclusion_effect)), max(effect_limits[1], np.nanmax(exclusion_effect))]
    for row in without_one:
        curve_ax.plot(bin_centers, row, color='#b0b0b0', linewidth=0.5, alpha=0.7, zorder=1)
    curve_ax.plot(bin_centers, full_mean, '-o', color=color, markersize=4, markeredgecolor='white', markeredgewidth=0.6, zorder=3)
    curve_ax.set_title(f'$\\bf{{({"ab"[panel]})}}$ {polarity_labels[polarity]} (n = {len(table)})', loc='left')
    curve_ax.grid(axis='y', color='#e5e5e5', linewidth=0.5)
    curve_ax.set_axisbelow(True)
    effect_ax.axhline(0, color='#999999', linewidth=0.6, zorder=0)
    for age_bin in range(N_AGE_BINS):
        values = exclusion_effect[:, age_bin]
        values = values[np.isfinite(values)]
        jitter = rng.uniform(-0.04, 0.04, len(values))
        effect_ax.scatter(bin_centers[age_bin] + jitter, values, color=color, s=9, alpha=0.45, linewidths=0, zorder=2)
    effect_ax.set_title(f'$\\bf{{({"cd"[panel]})}}$ Effect of being excluded on bin mean', loc='left')
    effect_ax.set_xlabel('Fraction of observed track')
    effect_ax.set_xlim(0, 1)
    effect_ax.xaxis.set_ticks(np.linspace(0, 1, N_AGE_BINS + 1))
    effect_ax.grid(axis='y', color='#e5e5e5', linewidth=0.5)
    effect_ax.set_axisbelow(True)
    full_difference = full_mean[-1] - full_mean[0]
    difference_effect = (without_one[:, -1] - without_one[:, 0]) - full_difference
    order = np.argsort(difference_effect * np.sign(full_difference))
    drop_row = {'polarity': polarity, 'largest_exclusion_effect': np.nanmax(np.abs(exclusion_effect))}
    for n_dropped in range(4):
        remaining = np.delete(matrix, order[:n_dropped], axis=0)
        drop_row[f'late_minus_early_dropping_{n_dropped}'] = np.nanmean(remaining[:, -1]) - np.nanmean(remaining[:, 0])
    drop_rows.append(drop_row)
for row_axes, low_high in ((loo_axes[0], curve_limits), (loo_axes[1], effect_limits)):
    ticks = cast(np.ndarray, MaxNLocator(nbins=5, steps=[1, 2, 2.5, 5, 10]).tick_values(*low_high))
    for ax in row_axes:
        cast(Axes, ax).yaxis.set_ticks(ticks)
        cast(Axes, ax).set_ylim(ticks[0], ticks[-1])
loo_axes[0, 0].set_ylabel('Interior CHL (mg m$^{-3}$)')
loo_axes[1, 0].set_ylabel('Change in bin mean (mg m$^{-3}$)')
plt.show()
drop_table = pd.DataFrame(drop_rows).set_index('polarity')
display(drop_table.round(3))

In [ ]:
periods = analysis.loc[
    analysis['age_frac'].lt(0.2) | analysis['age_frac'].ge(0.8),
    identity_columns + ['age_frac', 'date', 'CHL'],
].copy()
periods['period'] = np.where(periods['age_frac'].lt(0.2), 'Early', 'Late')
paired = periods.groupby(identity_columns + ['period'])['CHL'].mean().unstack('period').dropna()
paired['change'] = paired['Late'] - paired['Early']
period_dates = periods.groupby(identity_columns + ['period'])['date'].mean().unstack('period')
day_of_year = Normalize(1, 366)
season_cmap = plt.get_cmap('twilight')
gradient = np.linspace(0, 1, 25)
rng = np.random.default_rng(RANDOM_SEED)
bootstrap_intervals = {}
change_rows = []

paired_fig = plt.figure(figsize=(6.69, 5.1), layout='constrained')
grid = paired_fig.add_gridspec(2, 3, height_ratios=(2.1, 1), width_ratios=(1, 1, 0.03))
paired_left_ax = paired_fig.add_subplot(grid[0, 0])
paired_axes = np.array([paired_left_ax, paired_fig.add_subplot(grid[0, 1], sharey=paired_left_ax)])
change_ax = paired_fig.add_subplot(grid[1, :2])
for panel, polarity in enumerate(polarity_names):
    ax = cast(Axes, paired_axes[panel])
    values = paired.loc[polarity]
    color = polarity_colors[polarity]
    x = np.array([0, 1])
    dates = period_dates.loc[polarity].loc[values.index]
    segments = []
    segment_days = []
    for chl, date in zip(values[['Early', 'Late']].itertuples(index=False), dates[['Early', 'Late']].itertuples(index=False)):
        points = np.column_stack([gradient, chl[0] + gradient * (chl[1] - chl[0])])  # (25, 2)
        segments.extend(np.stack([points[:-1], points[1:]], axis=1))
        segment_days.extend((date[0] + pd.to_timedelta(gradient[:-1] * (date[1] - date[0]).days, unit='D')).dayofyear)
        ax.scatter(x, chl, color=season_cmap(day_of_year([date[0].dayofyear, date[1].dayofyear])), s=7, alpha=0.6, linewidths=0, zorder=2)
    ax.add_collection(LineCollection(segments, colors=season_cmap(day_of_year(np.array(segment_days))), linewidths=0.8, alpha=0.55, zorder=1))
    changes = values['change'].to_numpy()
    bootstrap = rng.choice(changes, size=(N_BOOTSTRAP, len(changes)), replace=True).mean(axis=1)
    bootstrap_intervals[polarity] = np.quantile(bootstrap, [0.025, 0.975])
    early_mean = values['Early'].mean()
    late_mean = values['Late'].mean()
    ax.plot(x, [early_mean, late_mean], '-o', color='#111111', linewidth=1.5, markersize=5, markerfacecolor='white', markeredgewidth=1, zorder=4)
    ax.xaxis.set_ticks(x)
    ax.xaxis.set_ticklabels(['First fifth', 'Last fifth'])
    ax.set_xlim(-0.25, 1.25)
    ax.set_title(f'$\\bf{{({"ab"[panel]})}}$ {polarity_labels[polarity]} (n = {len(values)})', loc='left')
    ax.grid(axis='y', color='#e5e5e5', linewidth=0.5)
    ax.set_axisbelow(True)
    same_sign = int((np.sign(changes) == np.sign(changes.mean())).sum())
    change_rows.append({
        'polarity': polarity, 'n_eddies': len(changes), 'mean': changes.mean(),
        'ci_low': bootstrap_intervals[polarity][0], 'ci_high': bootstrap_intervals[polarity][1],
        'median': np.median(changes), 'share_with_mean_sign': same_sign / len(changes),
        'sign_test_p': binomtest(same_sign, len(changes)).pvalue,
        'wilcoxon_p': wilcoxon(changes).pvalue,  # pyright: ignore[reportAttributeAccessIssue]
    })
paired_axes[0].set_ylabel('Interior CHL (mg m$^{-3}$)')
season_bar = paired_fig.colorbar(ScalarMappable(norm=day_of_year, cmap=season_cmap), cax=paired_fig.add_subplot(grid[0, 2]))
season_bar.set_ticks([1, 91, 182, 274, 365], labels=['Jan', 'Apr', 'Jul', 'Oct', 'Dec'])
season_bar.ax.tick_params(length=2)
season_bar.outline.set_linewidth(0.6)
change_ax.axhline(0, color='#999999', linewidth=0.6, zorder=0)
for position, polarity in enumerate(polarity_names):
    changes = paired.loc[polarity, 'change'].to_numpy()
    color = polarity_colors[polarity]
    jitter = rng.uniform(-0.09, 0.09, len(changes))
    change_ax.scatter(position + jitter, changes, color=color, s=13, alpha=0.45, linewidths=0, zorder=2)
    change_ax.hlines(np.median(changes), position - 0.1, position + 0.1, color=color, linewidth=1.3, zorder=3)
    mean = changes.mean()
    low, high = bootstrap_intervals[polarity]
    change_ax.errorbar(position, mean, yerr=[[mean - low], [high - mean]], fmt='o', color='#111111', markerfacecolor='white', markersize=5, capsize=3, elinewidth=1.1, zorder=4)
change_ax.set_xticks(range(len(polarity_names)), [polarity_labels[polarity] for polarity in polarity_names])
change_ax.set_xlim(-0.45, 1.45)
change_ax.set_ylabel('$\\Delta$CHL, last fifth - first fifth (mg m$^{-3}$)')
change_ax.set_title('$\\bf{(c)}$ Paired change for each eddy', loc='left')
change_ax.grid(axis='y', color='#e5e5e5', linewidth=0.5)
change_ax.set_axisbelow(True)
change_ax.errorbar([], [], yerr=[], fmt='o', color='#111111', markerfacecolor='white', markersize=5, capsize=3, elinewidth=1.1, label='Mean with 95% bootstrap interval')
change_ax.hlines([], [], [], color='#555555', linewidth=1.3, label='Median')
change_ax.legend(loc='upper center')
for ax, column in ((paired_axes[0], ['Early', 'Late']), (change_ax, ['change'])):
    ticks = cast(np.ndarray, MaxNLocator(nbins=5, steps=[1, 2, 2.5, 5, 10]).tick_values(paired[column].min().min(), paired[column].max().max()))
    cast(Axes, ax).yaxis.set_ticks(ticks)
    cast(Axes, ax).set_ylim(ticks[0], ticks[-1])
plt.show()
change_table = pd.DataFrame(change_rows).set_index('polarity')
display(change_table.round(3))

In [ ]:
track_frames = []
for polarity in polarity_names:
    tracked = TrackEddiesObservations.load_file(str(DATA_DIR / f'silver/eddy_track/{polarity}/{polarity}_tracks.zarr'))
    keep = ~tracked.virtual.astype(bool)
    track_frames.append(pd.DataFrame({
        'polarity': polarity, 'track_id': tracked.track[keep].astype(int),
        'date': pd.Timestamp(PET_EPOCH) + pd.to_timedelta(tracked.time[keep].astype(int), unit='D'),
        'contour_speed': tracked.speed_average[keep],
        'radius_km': tracked.radius_s[keep] / 1000,
    }))
tracks = pd.concat(track_frames).merge(
    analysis.drop_duplicates(identity_columns)[identity_columns + ['birth_date', 'death_date']], on=identity_columns,
)
tracks['age_frac'] = ((tracks['date'] - tracks['birth_date']) / (tracks['death_date'] - tracks['birth_date'])).clip(0, 1)
tracks['age_bin'] = np.minimum((tracks['age_frac'] * N_AGE_BINS).astype(int), N_AGE_BINS - 1)
track_bins = tracks.groupby(identity_columns + ['age_bin'])[['contour_speed', 'radius_km']].mean()
rng = np.random.default_rng(RANDOM_SEED)
dynamics_fig, dynamics_axes = cast(tuple[Figure, np.ndarray], plt.subplots(1, 2, figsize=(6.69, 2.9), layout='constrained'))
for panel, (metric, label) in enumerate((('contour_speed', 'Speed along the speed contour (m s$^{-1}$)'), ('radius_km', 'Speed radius (km)'))):
    ax = cast(Axes, dynamics_axes[panel])
    limits = []
    for offset, polarity in zip((-0.012, 0.012), polarity_names):
        matrix = track_bins.loc[polarity, metric].unstack('age_bin').reindex(columns=range(N_AGE_BINS)).to_numpy(dtype=float)  # (n_eddies, n_bins)
        means = np.nanmean(matrix, axis=0)
        sampled = matrix[rng.integers(0, len(matrix), size=(N_BOOTSTRAP, len(matrix)))]  # (n_bootstrap, n_eddies, n_bins)
        low, high = np.quantile(np.nanmean(sampled, axis=1), [0.025, 0.975], axis=0)
        limits.extend([low.min(), high.max()])
        color = polarity_colors[polarity]
        ax.errorbar(bin_centers + offset, means, yerr=[means - low, high - means], fmt='none', ecolor=color, capsize=2, elinewidth=0.8, capthick=0.8, zorder=2)
        ax.plot(bin_centers + offset, means, '-o', color=color, markersize=4, markeredgecolor='white', markeredgewidth=0.6, label=f'{polarity_labels[polarity]} (n = {len(matrix)})', zorder=3)
    ax.set_title(f'$\\bf{{({"ab"[panel]})}}$ {label}', loc='left')
    ax.set_xlabel('Fraction of observed track')
    ax.xaxis.set_ticks(np.linspace(0, 1, N_AGE_BINS + 1))
    ax.set_xlim(0, 1)
    ax.grid(axis='y', color='#e5e5e5', linewidth=0.5)
    ax.set_axisbelow(True)
    ticks = cast(np.ndarray, MaxNLocator(nbins=5, steps=[1, 2, 2.5, 5, 10]).tick_values(min(limits), max(limits)))
    ax.yaxis.set_ticks(ticks)
    ax.set_ylim(ticks[0], ticks[-1])
dynamics_fig.legend(*dynamics_axes[0].get_legend_handles_labels(), loc='outside lower center', ncol=2)
plt.show()

Show that something is changing within the eddy rather than the eddies getting older while season is changing (e.g., cyclones that we see happen to have decreased CHL because it enters winter, and anticyclones that we see happen to have increased CHL because it enters spring).

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter('ignore', ConvergenceWarning)
    warnings.simplefilter('ignore', SingularMatrixWarning)
    unadjusted_model = smf.mixedlm(
        'CHL ~ age_frac * is_anticyclone', analysis, groups=analysis['eddy_key'], re_formula='1',
    ).fit(reml=True, method='lbfgs', disp=False)
    adjusted_model = smf.mixedlm(
        'CHL ~ age_frac * is_anticyclone + season_sin + season_cos', analysis, groups=analysis['eddy_key'], re_formula='1',
    ).fit(reml=True, method='lbfgs', disp=False)

def predict_fixed_effects(model, frame):
    names = model.fe_params.index
    terms = {
        'Intercept': np.ones(len(frame)),
        'age_frac': frame['age_frac'].to_numpy(dtype=float),
        'is_anticyclone': frame['is_anticyclone'].to_numpy(dtype=float),
        'age_frac:is_anticyclone': (frame['age_frac'] * frame['is_anticyclone']).to_numpy(dtype=float),
        'season_sin': frame['season_sin'].to_numpy(dtype=float),
        'season_cos': frame['season_cos'].to_numpy(dtype=float),
    }
    matrix = np.column_stack([terms[name] for name in names])
    covariance = model.cov_params().loc[names, names].to_numpy(dtype=float)
    mean = matrix @ model.fe_params.to_numpy(dtype=float)
    standard_error = np.sqrt(np.einsum('ij,jk,ik->i', matrix, covariance, matrix))
    return mean, mean - 1.96 * standard_error, mean + 1.96 * standard_error

age_grid = np.linspace(0, 1, 101)
season_sin_reference = analysis['season_sin'].mean()
season_cos_reference = analysis['season_cos'].mean()
trend_fig, trend_axes = cast(tuple[Figure, np.ndarray], plt.subplots(
    2, 2, figsize=(6.69, 5.0), sharex=True, layout='constrained',
    gridspec_kw={'height_ratios': (2.1, 1)},
))
trend_limits = []
for panel, polarity in enumerate(polarity_names):
    trend_ax = cast(Axes, trend_axes[0, panel])
    season_ax = cast(Axes, trend_axes[1, panel])
    prediction = pd.DataFrame({
        'age_frac': age_grid,
        'is_anticyclone': int(polarity == 'anticyclone'),
        'season_sin': season_sin_reference,
        'season_cos': season_cos_reference,
    })
    raw_mean, _, _ = predict_fixed_effects(unadjusted_model, prediction)
    adjusted_mean, adjusted_low, adjusted_high = predict_fixed_effects(adjusted_model, prediction)
    color = polarity_colors[polarity]
    trend_ax.plot(age_grid, raw_mean, linestyle=(0, (4, 2.5)), color='#555555', linewidth=1.1)
    trend_ax.fill_between(age_grid, adjusted_low, adjusted_high, color=color, alpha=0.16, linewidth=0)
    trend_ax.plot(age_grid, adjusted_mean, color=color, linewidth=1.6)
    trend_ax.set_title(f'$\\bf{{({"ab"[panel]})}}$ {polarity_labels[polarity]}', loc='left')
    trend_ax.grid(axis='y', color='#e5e5e5', linewidth=0.5)
    trend_ax.set_axisbelow(True)
    trend_limits.extend([adjusted_low.min(), adjusted_high.max(), raw_mean.min(), raw_mean.max()])
    polarity_data = analysis.loc[analysis['polarity'].eq(polarity)]
    season_ax.scatter(
        polarity_data['age_frac'], polarity_data['date'].dt.dayofyear,
        color=color, s=7, alpha=0.25, linewidths=0, rasterized=True,
    )
    season_ax.set_xlabel('Fraction of observed track')
    season_ax.set_xlim(0, 1)
    season_ax.set_ylim(0, 367)
    season_ax.yaxis.set_ticks([1, 91, 182, 274, 365])
    season_ax.yaxis.set_ticklabels(['Jan', 'Apr', 'Jul', 'Oct', 'Dec'])
    season_ax.grid(color='#e5e5e5', linewidth=0.5)
    season_ax.set_axisbelow(True)
padding = 0.08 * (max(trend_limits) - min(trend_limits))
for trend_ax in trend_axes[0]:
    cast(Axes, trend_ax).set_ylim(min(trend_limits) - padding, max(trend_limits) + padding)
trend_axes[0, 0].set_ylabel('Predicted interior CHL (mg m$^{-3}$)')
trend_axes[1, 0].set_ylabel('Day of year')
trend_axes[0, 0].plot([], [], linestyle=(0, (4, 2.5)), color='#555555', linewidth=1.1, label='Unadjusted')
trend_axes[0, 0].plot([], [], color='#111111', linewidth=1.6, label='Season-adjusted')
trend_fig.legend(*trend_axes[0, 0].get_legend_handles_labels(), loc='outside lower center', ncol=2)
plt.show()